### Genetic algorithm


In [21]:
!pip install scikit-learn-extra


  Using cached scikit-learn-extra-0.3.0.tar.gz (818 kB)
  Installing build dependencies: started
  Installing build dependencies: finished with status 'done'
  Getting requirements to build wheel: started
  Getting requirements to build wheel: finished with status 'done'
  Preparing metadata (pyproject.toml): started
  Preparing metadata (pyproject.toml): finished with status 'done'
Failed to build scikit-learn-extra


  error: subprocess-exited-with-error
  
  exit code: 1
  
  [77 lines of output]
  C:\Users\mazen\AppData\Local\Temp\pip-build-env-vdz8njrm\overlay\Lib\site-packages\setuptools\dist.py:599: SetuptoolsDeprecationWarning: Invalid dash-separated key 'description-file' in 'metadata' (setup.cfg), please use the underscore name 'description_file' instead.
  !!
  
          ********************************************************************************
          Usage of dash-separated 'description-file' will not be supported in future
          versions. Please use the underscore name 'description_file' instead.
          (Affected: scikit-learn-extra).
  
          Available configuration options are listed in:
          https://setuptools.pypa.io/en/latest/userguide/declarative_config.html
  
          This deprecation is overdue, please update your project and remove deprecated
          calls to avoid build errors in the future.
  
          See https://github.com/pypa/setuptools/discu

In [22]:
import random
import numpy as np
import pandas as pd
from sklearn.preprocessing import StandardScaler
from sklearn_extra.cluster import KMedoids
from sklearn.metrics import silhouette_score

ModuleNotFoundError: No module named 'sklearn_extra'

In [ ]:
df = pd.read_csv('cleaned_data.csv')

# Apply scaling
numerical_cols = [
    'person_age',
    'person_income',
    'person_emp_length',
    'loan_amnt',
    'loan_int_rate',
    'loan_percent_income',
    'cb_person_cred_hist_length'
]

scaler = StandardScaler()
df[numerical_cols] = scaler.fit_transform(df[numerical_cols])

# Separate features and target
X = df.drop('loan_status', axis=1)
y = df['loan_status']
feature_names = list(X.columns)

# Sample to match K-Medoids task speed
X_sample = X.sample(n=5000, random_state=42)

print("Available Features:", feature_names)
print("Total Features:", len(feature_names))

# GA Parameters
population_size = 10
mutation_probability = 0.2
generations = 10

print("\nGenetic Algorithm Parameters:")
print("Population:", population_size)
print("Mutation probability:", mutation_probability)
print("Generations:", generations, "\n")

In [ ]:
# Generate a random individual
def create_individual():
    return [random.randint(0, 1) for _ in range(len(feature_names))]

In [ ]:
# Calculate fitness (silhouette score of K-Medoids with selected features)
def fitness(individual):
    selected = [i for i in range(len(individual)) if individual[i] == 1]
    if len(selected) < 2:
        return 0

    data = X_sample.iloc[:, selected].values

    model = KMedoids(
        n_clusters=3,
        metric='manhattan',
        init='k-medoids++',
        random_state=42
    )
    labels = model.fit_predict(data)

    return silhouette_score(data, labels)

In [ ]:
# Select parents using tournament selection
def select_parent(population):
    tournament = random.sample(population, 3)
    return max(tournament, key=lambda x: fitness(x))

In [ ]:
# Single-point crossover
def crossover(parent1, parent2):
    point = random.randint(1, len(parent1) - 1)
    child = parent1[:point] + parent2[point:]
    return child

In [ ]:
# Mutation
def mutate(individual):
    for i in range(len(individual)):
        if random.random() < mutation_probability:
            individual[i] = 1 - individual[i]
    return individual

In [18]:
# Main Genetic Algorithm
def genetic_algorithm():
    population = [create_individual() for _ in range(population_size)]

    for generation in range(generations):
        population = sorted(population, key=lambda x: fitness(x), reverse=True)

        best = population[0]
        best_fitness = fitness(best)
        selected = [feature_names[i] for i in range(len(best)) if best[i] == 1]
        print(f"Generation {generation + 1}:")
        print(f"Best Chromosome: {best}")
        print(f"Selected Features: {selected}")
        print(f"Fitness (Silhouette Score): {best_fitness:.4f}")
        print("------")

        new_population = []
        for _ in range(population_size):
            parent1 = select_parent(population)
            parent2 = select_parent(population)
            child = crossover(parent1, parent2)
            child = mutate(child)
            new_population.append(child)

        population = new_population

    best = max(population, key=lambda x: fitness(x))
    best_fitness = fitness(best)
    selected = [feature_names[i] for i in range(len(best)) if best[i] == 1]

    print("\nFinal Solution:")
    print(f"Best Chromosome: {best}")
    print(f"Selected Features: {selected}")
    print(f"Number of Features: {sum(best)} out of {len(feature_names)}")
    print(f"Silhouette Score: {best_fitness:.4f}")
    return best, best_fitness

# Baseline
baseline_individual = [1] * len(feature_names)
baseline_score = fitness(baseline_individual)
print(f"Baseline Silhouette Score (All Features): {baseline_score:.4f}\n")


best_solution, best_score = genetic_algorithm()

print(f"\n--- Comparison ---")
print(f"Baseline (All Features) Silhouette: {baseline_score:.4f}")
print(f"GA Optimized Silhouette:            {best_score:.4f}")
print(f"Improvement:                        {(best_score - baseline_score):.4f}")
print(f"Features Reduced: {len(feature_names)} → {sum(best_solution)}")

ModuleNotFoundError: No module named 'sklearn_extra'